In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interactive
from IPython.display import display, HTML

# 1. MathJax & CSS configuration
display(HTML("""
<script>
    if (window.MathJax) {
        MathJax.Hub.Config({
            "HTML-CSS": { scale: 130 },
            SVG: { scale: 130 }
        });
    }
</script>
<style>
    .rendered_html math { font-size: 1.3em !important; }
    .output_subarea, .jp-OutputArea-output { max-height: none !important; }
</style>
"""))

sp.init_printing(use_latex='mathjax')

# 2. Strict symbolic computation
omega, alpha_sym = sp.symbols('omega alpha', real=True)
n = sp.symbols('n', integer=True)
q = sp.symbols('q')

# Symbolic geometric series
X_q = sp.summation(q**n, (n, 0, sp.oo))

# Select the convergent branch and substitute q = alpha*exp(-j*omega)
X_ejw_simplified = sp.simplify(
    X_q.args[0][0].subs(
        q,
        alpha_sym * sp.exp(-sp.I * omega)
    )
)

# Symbolic magnitude obtained from X(e^{jω}) * X*(e^{jω})
X_conjugate = sp.conjugate(X_ejw_simplified)

mag_squared = sp.trigsimp(
    sp.simplify(
        sp.expand_complex(
            X_ejw_simplified * X_conjugate
        )
    )
)

mag_sym = sp.powdenest(
    sp.sqrt(mag_squared),
    force=True
)

mag_sym = sp.simplify(mag_sym)

# Symbolic phase
phase_sym = -sp.atan(
    (alpha_sym * sp.sin(omega)) /
    (1 - alpha_sym * sp.cos(omega))
)

# Energy density spectrum obtained symbolically from |X|^2
energy_sym = sp.simplify(mag_squared)

# Printing the symbolic results
display(HTML("<h3 style='color: #2c3e50;'>Strict Symbolic Derivation via SymPy:</h3>"))

display(sp.Eq(
    sp.Symbol(r'\mathcal{X}(e^{j\omega})'),
    X_ejw_simplified
))

display(sp.Eq(
    sp.Symbol(r'|\mathcal{X}(e^{j\omega})|'),
    mag_sym
))

display(sp.Eq(
    sp.Symbol(r'\angle \mathcal{X}(e^{j\omega})'),
    phase_sym
))

display(sp.Eq(
    sp.Symbol(r'S_{xx}(\omega)'),
    energy_sym
))

# 3. Interactive plotting function
def plot_book_figures(alpha_pos, alpha_neg):
    omega_2pi = np.linspace(-2 * np.pi, 2 * np.pi, 1200)
    omega_pi = np.linspace(-np.pi, np.pi, 600)

    # --- Case (a): alpha > 0 ---
    amp_pos = 1.0 / np.sqrt(
        1.0 - 2.0 * alpha_pos * np.cos(omega_2pi) + alpha_pos**2
    )
    phase_pos = -np.arctan2(
        alpha_pos * np.sin(omega_2pi),
        1.0 - alpha_pos * np.cos(omega_2pi)
    )
    energy_pos = 1.0 / (
        1.0 - 2.0 * alpha_pos * np.cos(omega_pi) + alpha_pos**2
    )

    # --- Case (b): alpha < 0 ---
    amp_neg = 1.0 / np.sqrt(
        1.0 - 2.0 * alpha_neg * np.cos(omega_2pi) + alpha_neg**2
    )
    phase_neg = -np.arctan2(
        alpha_neg * np.sin(omega_2pi),
        1.0 - alpha_neg * np.cos(omega_2pi)
    )
    energy_neg = 1.0 / (
        1.0 - 2.0 * alpha_neg * np.cos(omega_pi) + alpha_neg**2
    )

    fig, axes = plt.subplots(3, 2, figsize=(13, 10))

    # --- Figure (a): Alpha > 0 ---
    axes[0, 0].plot(
        omega_2pi / np.pi,
        amp_pos,
        color='red',
        lw=2
    )
    axes[0, 0].set_ylabel(
        r'$|X(e^{j\omega})|$',
        fontsize=11
    )
    axes[0, 0].set_title(
        r'(a) Amplitude Spectrum ($\alpha = %.2f > 0$)' % alpha_pos,
        fontsize=11
    )
    axes[0, 0].grid(True, linestyle='--', alpha=0.5)

    axes[1, 0].plot(
        omega_2pi / np.pi,
        phase_pos,
        color='red',
        lw=2
    )
    axes[1, 0].set_ylabel(
        r'$\angle X(e^{j\omega})$',
        fontsize=11
    )
    axes[1, 0].set_title(
        r'(a) Phase Spectrum ($\alpha > 0$)',
        fontsize=11
    )
    axes[1, 0].grid(True, linestyle='--', alpha=0.5)

    # --- Figure (b): Alpha < 0 ---
    axes[0, 1].plot(
        omega_2pi / np.pi,
        amp_neg,
        color='red',
        lw=2
    )
    axes[0, 1].set_title(
        r'(b) Amplitude Spectrum ($\alpha = %.2f < 0$)' % alpha_neg,
        fontsize=11
    )
    axes[0, 1].grid(True, linestyle='--', alpha=0.5)

    axes[1, 1].plot(
        omega_2pi / np.pi,
        phase_neg,
        color='red',
        lw=2
    )
    axes[1, 1].set_ylabel(
        r'$\angle X(e^{j\omega})$',
        fontsize=11
    )
    axes[1, 1].set_title(
        r'(b) Phase Spectrum ($\alpha < 0$)',
        fontsize=11
    )
    axes[1, 1].grid(True, linestyle='--', alpha=0.5)

    # --- Figure (c): Energy Density Spectrum Sxx(ω) ---
    axes[2, 0].plot(
        omega_pi / np.pi,
        energy_pos,
        color='red',
        lw=2
    )
    axes[2, 0].set_xlabel(
        r'$\omega$',
        fontsize=11
    )
    axes[2, 0].set_ylabel(
        r'$S_{xx}(\omega)$',
        fontsize=11
    )
    axes[2, 0].set_title(
        r'(c) Energy Density Spectrum ($\alpha > 0$)',
        fontsize=11
    )
    axes[2, 0].set_xticks([-1, -0.5, 0, 0.5, 1])
    axes[2, 0].set_xticklabels([
        r'$-\pi$',
        r'$-\pi/2$',
        '0',
        r'$\pi/2$',
        r'$\pi$'
    ])
    axes[2, 0].set_xlim(-1, 1)
    axes[2, 0].grid(True, linestyle='--', alpha=0.5)

    axes[2, 1].plot(
        omega_pi / np.pi,
        energy_neg,
        color='red',
        lw=2
    )
    axes[2, 1].set_xlabel(
        r'$\omega$',
        fontsize=11
    )
    axes[2, 1].set_ylabel(
        r'$S_{xx}(\omega)$',
        fontsize=11
    )
    axes[2, 1].set_title(
        r'(c) Energy Density Spectrum ($\alpha < 0$)',
        fontsize=11
    )
    axes[2, 1].set_xticks([-1, -0.5, 0, 0.5, 1])
    axes[2, 1].set_xticklabels([
        r'$-\pi$',
        r'$-\pi/2$',
        '0',
        r'$\pi/2$',
        r'$\pi$'
    ])
    axes[2, 1].set_xlim(-1, 1)
    axes[2, 1].grid(True, linestyle='--', alpha=0.5)

    # Common x-axis settings for amplitude and phase
    for row in [0, 1]:
        for col in [0, 1]:
            axes[row, col].set_xticks([-2, -1, 0, 1, 2])
            axes[row, col].set_xticklabels([
                r'$-2\pi$',
                r'$-\pi$',
                '0',
                r'$\pi$',
                r'$2\pi$'
            ])
            axes[row, col].set_xlim(-2, 2)

    plt.tight_layout()
    plt.show()

# 4. Interactive widget with sliders
interactive_plot = interactive(
    plot_book_figures,
    alpha_pos=widgets.FloatSlider(
        value=0.6,
        min=0.01,
        max=0.95,
        step=0.05,
        description='Positive $\\alpha$:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='450px')
    ),
    alpha_neg=widgets.FloatSlider(
        value=-0.6,
        min=-0.95,
        max=-0.01,
        step=0.05,
        description='Negative $\\alpha$:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='450px')
    )
)

display(interactive_plot)